# Télécharger les librairies importantes

Pour les configurations locales, attention à bien gérer vos espaces virtuels

In [4]:
# Retirer les # avant de lancer
# !curl https://raw.githubusercontent.com/emilienschultz/slides-bertopic/refs/heads/main/material/requirements.txt > requirements.txt
# !pip install -r requirements.txt

# Aperçu de la pipeline et de son fonctionnement

## Lancer son premier topic model avec BERTopic

_Penser à importer les données dans votre espace de travail_

Commençons par ouvrir les données

In [5]:
import pandas as pd
df = pd.read_csv("./theses-soutenues-curated-stratified.csv")
df.head(5)

,year,CI,oai_set_specs,titres.en,resumes.en,lang_res.en,topics.en,titres.fr,resumes.fr,lang_res.fr,topics.fr,swapped,resumes.en.len,resumes.fr.len
0,2010.0,CI-32884,ddc:570||ddc:610,Study of epigenetic deregulation induced by BR...,"Nowadays, it is clear that cancer cannot be re...",EN,Chromatine,Dérégulation épigénétique induites par la prot...,Il apparait de nos jours évident que les cance...,FR,Épigénétique,NaN,1371,1675
1,2010.0,CI-9009,ddc:530,Synthesis and Study of Thermoelectric Material...,The development of new intermetallic compounds...,EN,NaN,Synthèse et Etude des Matériaux Thermoélectriq...,Le développement de nouveaux composés intermét...,FR,Composés intermétalliques,NaN,1390,1247
2,2010.0,CI-12137,ddc:004||ddc:620,Electromagnetic emissions analysis of integrat...,"In the area of secure integrated circuits, suc...",EN,NaN,Analyse des émissions électromagnétiques des c...,Dans le domaine de la sécurisation des circuit...,FR,Circuits intégrés||Contre-mesures électronique...,NaN,1308,1637
3,2010.0,CI-9599,ddc:960||ddc:940,The political evolution of Niger towards democ...,"As in the fifties, the access to independencie...",EN,NaN,Évolution du Niger indépendant vers le régime ...,En Afrique de la même manière qu’à la fin des ...,FR,Politique et gouvernement -- Niger -- 1960-1993,NaN,1108,1247
4,2010.0,CI-151911,ddc:530||ddc:540,Elaboration of functionalized nanoparticles : ...,Spinel structured iron oxide nanoparticles ope...,EN,Succimer,Elaboration de nanoparticules fonctionnelles :...,Les nanoparticules d’oxyde de fer de structure...,FR,"Imagerie par résonance magnétique||Citrique, A...",NaN,2153,2719


Créer le modèle et le lancer.

Pour commencer, on n'utilisera que 1000 exemples afin de limiter le temps de calcul

In [ ]:
from bertopic import BERTopic

docs = df["resumes.fr"].sample(1000).to_list()
topic_model = BERTopic(language="french")
topic_model.fit(documents=docs)
# ~2 min sur Colabs

# Version finale

In [10]:
from simplemma import lemmatize
from string import punctuation

class CustomLemmatizer:
    """An object to apply the lemmatize function."""

    def __init__(self, language, stop_words) -> None:
        self.__language = (language,)
        self.__stop_words = stop_words

    def __call__(self, doc: str) -> list[str]:
        """ """
        doc = doc.lower()
        doc = "".join([c for c in doc if c not in punctuation])
        out = []
        for word in doc.split(" "):
            if (word not in self.__stop_words) and (len(word) > 0):
                try:
                    lemma = lemmatize(word, lang=self.__language)
                except Exception as e:
                    # if lemmatization failed, skip it and use the word instead
                    lemma = word
                    print(f"BERTopic - Lemmatization failed (word: {word}) - Error : {e}")
                if lemma not in self.__stop_words:
                    out += [lemma]
        return out
    
from datasets import load_from_disk
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from stopwordsiso import stopwords
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
RANDOM_SEED = 2306406
np.random.seed(RANDOM_SEED)

language = "english" # or "french"
language_short = language[:2] # "en" or "fr"
ds = load_from_disk(f"./embeddings/gte-multilingual-base-en-SBERT")
docs = np.array(ds[f"resumes.en"]) # 6500 rows
embeddings = np.array(ds["embedding"])			 # Shape : 6500 x 768

vectorizer_model = CountVectorizer(
    tokenizer=CustomLemmatizer(language_short, list(stopwords(language_short)))
)

hdbscan_model = HDBSCAN(
    min_cluster_size=30, 
    min_samples=10,
    prediction_data=True
)

umap_model = UMAP(
    n_neighbors = 50,
    metric = "cosine",
    n_components = 8,
    min_dist=0.0,
    low_memory = False,
    random_state=RANDOM_SEED   
)

topic_model = BERTopic(
	language = language,
	vectorizer_model = vectorizer_model,
    umap_model= umap_model,
    hdbscan_model=hdbscan_model
)
topics, probabilities = topic_model.fit_transform(
    documents=docs, 
    embeddings=embeddings
)

umap_model_for_vis = UMAP(
    n_components = 2,
    
    n_neighbors = 50,
    metric = "cosine",
    min_dist=0.0,
    low_memory = False,
    random_state=RANDOM_SEED   
)

reduced_embeddings = umap_model_for_vis.fit_transform(embeddings)

topic_model.reduce_topics(docs=docs, nr_topics=10)

topic_model.visualize_documents(
    docs = docs,
    # topics = reduced_topics, 
    reduced_embeddings = reduced_embeddings,
    hide_annotations = True, 
    hide_document_hover = True,
    height = 600, 
    width = 1000
)

/opt/miniconda3/envs/bertopic/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/opt/miniconda3/envs/bertopic/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/opt/miniconda3/envs/bertopic/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/opt/miniconda3/envs/bertopic/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning:

The parameter 'token_pattern' will not be used since 'tokenizer' is not None'

/opt/miniconda3/envs/bertopic/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/opt/miniconda3/envs/bertopic/lib/python3.12/site-pa